# Jev vs Claude Opus: clinical coherence and calibration benchmark (v4)

Everything lives in `MyDrive/jev_bench/`. Code for this version is in `code/v4/` and is checked against fixed SHA-256 hashes before anything runs. See `CHANGELOG.md` and `code/v4/CHANGES.md` for version history.

**New in v4** (all logged as deviations in the pre-registration, 2026-09-27): matched H1 comparison (D1); fixed Opus sampled-answer parser, rerun as a new arm `opus_sampled_v2` (D2); exploratory H4N, vague inputs with "none of these" offered (D3); fingerprinting skips Google-native files such as `.gsheet`. Running this notebook adds only the new jobs; everything already in `results.jsonl` is kept and skipped.

**Secrets** (key icon, left sidebar, notebook access on): `OPENROUTER_API_KEY`, `ANTHROPIC_API_KEY`.

Run top to bottom. Sections 6 to 8 skip themselves until your data files are in `jev_bench/data/`.

## 1. Setup

In [1]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, sys, shutil, hashlib
VERSION = 'v4'
WORK = '/content/drive/MyDrive/jev_bench'
CODE = f'{WORK}/code/{VERSION}'
D = f'{WORK}/data'
OUT = f'{WORK}/results.jsonl'
os.makedirs(D, exist_ok=True)

!pip -q install anthropic

for k in ('OPENROUTER_API_KEY', 'ANTHROPIC_API_KEY'):
    try:
        os.environ[k] = userdata.get(k)
        print(k, 'loaded')
    except Exception as e:
        print(k, 'MISSING:', e)

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 18.9 MB/s eta 0:00:00
OPENROUTER_API_KEY loaded
ANTHROPIC_API_KEY loaded


## 2. Load and verify the harness code
Copies `code/v4/` into the session and refuses to continue if any file differs from the version's recorded hash.

In [2]:
EXPECTED = {'clients.py': 'c5305dc7960b919efc028e926ef125ff2553c5b3f9e825a4563a910a082b351a', 'run.py': 'd07c12d3384e23d05bef119f508d09165581f80405d68e06aeea5147da28c442', 'analyze.py': '1015735f0e3cb8ef5e5b1c5b35f5c087c313bcebc718c0348aca70a4a8fdef8e', 'ddxplus_to_cases.py': '0a2c9072ec480fe52e3eef4f420fcc2f37a71372d252c84a442e34a922912c51'}

def sha(path):
    return hashlib.sha256(open(path, 'rb').read()).hexdigest()

RUN = '/content/jev_bench'
os.makedirs(RUN, exist_ok=True)
bad = []
for f, h in EXPECTED.items():
    p = f'{CODE}/{f}'
    if not os.path.exists(p):
        bad.append(f'{f}: missing')
    elif sha(p) != h:
        bad.append(f'{f}: hash mismatch ({sha(p)[:12]} != {h[:12]})')
    else:
        shutil.copy(p, f'{RUN}/{f}')
if bad:
    raise SystemExit('Code check FAILED, paste this to Claude:\n' + '\n'.join(bad))
os.chdir(RUN)
sys.path.insert(0, RUN)
print('All', len(EXPECTED), 'files verified for', VERSION)

All 4 files verified for v4


## 3. Version control

`MyDrive/jev_bench/` is a git repository. `snapshot(label)` records a manifest (code hashes, data-file hashes, results hash), saves a snapshot of this notebook as it ran, commits, and sets `HARNESS_COMMIT` so every result row carries it. Data files are hashed but never committed. A plain timestamped copy of each manifest also goes to `versions/`.

In [3]:
import json, subprocess, datetime, glob

def _git(*args):
    return subprocess.run(['git', '-C', WORK, *args], capture_output=True, text=True)

GOOGLE_NATIVE = ('.gsheet', '.gdoc', '.gslides', '.gform', '.gdraw', '.gmap', '.gsite', '.gjam')

def _safe_sha(p):
    try:
        return sha(p)
    except OSError as e:
        return f'unreadable: {e.strerror}'

def _notebook_json():
    try:
        from google.colab import _message
        return _message.blocking_request('get_ipynb', request='', timeout_sec=30)['ipynb']
    except Exception as e:
        print('  (could not capture notebook contents:', e, ')')
        return None

def snapshot(label):
    stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    os.makedirs(f'{WORK}/runs', exist_ok=True)
    nbj = _notebook_json()
    if nbj:
        json.dump(nbj, open(f'{WORK}/runs/notebook_snapshot.ipynb', 'w'), indent=1)
    manifest = {'label': label, 'utc': stamp, 'version': VERSION,
                'code_sha256': {f: sha(f'{CODE}/{f}') for f in EXPECTED},
                'data_sha256': {os.path.relpath(p, WORK): _safe_sha(p)
                                for p in sorted(glob.glob(f'{D}/**/*', recursive=True))
                                if os.path.isfile(p) and not p.endswith(GOOGLE_NATIVE)},
                'results_sha256': sha(OUT) if os.path.exists(OUT) else None}
    json.dump(manifest, open(f'{WORK}/runs/MANIFEST.json', 'w'), indent=2)
    vdir = f'{WORK}/versions/{stamp}_{label}'
    os.makedirs(vdir, exist_ok=True)
    shutil.copy(f'{WORK}/runs/MANIFEST.json', vdir)

    if not os.path.exists(f'{WORK}/.git'):
        _git('init', '-q')
        _git('config', 'user.name', 'jev_bench')
        _git('config', 'user.email', 'jev_bench@localhost')
        _git('config', 'core.fileMode', 'false')
        open(f'{WORK}/.gitignore', 'w').write('data/\nversions/\n')
    _git('add', '-A')
    c = _git('commit', '-q', '-m', f'{VERSION} {label} {stamp}')
    head = _git('rev-parse', '--short', 'HEAD').stdout.strip()
    if head:
        os.environ['HARNESS_COMMIT'] = head
        print(f'snapshot "{label}": commit {head}' + ('' if c.returncode == 0 else ' (no changes)'))
    else:
        h = hashlib.sha256(''.join(manifest['code_sha256'].values()).encode()).hexdigest()[:10]
        os.environ['HARNESS_COMMIT'] = f'nogit-{h}'
        print(f'snapshot "{label}": git unavailable, using code hash {h}')

snapshot('setup')
print(_git('log', '--oneline', '-5').stdout)

snapshot "setup": commit 73c2c48
73c2c48 v4 setup 20260927T040752Z
16cc91e v3 analysis 20260927T034908Z
6cb2181 v3 pre-run-opus-sampled 20260927T031025Z
ef4e15e v3 pre-run-opus-verbalized 20260927T021848Z
9ab7e87 v3 pre-run-jev 20260927T021625Z



## 4. Offline smoke test (no API calls, no cost)

In [4]:
%%writefile example_options.json
{
  "toy_eye": {
    "options": {
      "retinal_detachment": "Retinal detachment",
      "posterior_vitreous_detachment": "Posterior vitreous detachment",
      "acute_angle_closure_glaucoma": "Acute angle-closure glaucoma",
      "central_retinal_artery_occlusion": "Central retinal artery occlusion",
      "bacterial_conjunctivitis": "Bacterial conjunctivitis",
      "anterior_uveitis": "Anterior uveitis",
      "corneal_abrasion": "Corneal abrasion",
      "dry_eye_disease": "Dry eye disease"
    },
    "distractors": {
      "ingrown_toenail": "Ingrown toenail",
      "benign_prostatic_hyperplasia": "Benign prostatic hyperplasia",
      "plantar_fasciitis": "Plantar fasciitis"
    }
  }
}


Writing example_options.json


In [5]:
%%writefile example_cases.csv
case_id,dataset,text,true_dx,paraphrase_1,paraphrase_2,paraphrase_3,distractors
t1,toy_eye,"Toy case: new flashes and a dark curtain coming over the vision in my right eye since yesterday.",retinal_detachment,"Toy case: since yesterday my right eye has flashing lights and a shade moving across.",,,ingrown_toenail;plantar_fasciitis
t2,toy_eye,"Toy case: sudden new floaters in my left eye, no change in vision otherwise, age 62.",posterior_vitreous_detachment,,,,ingrown_toenail
t3,toy_eye,"Toy case: severe eye pain, halos around lights, headache and vomiting tonight.",acute_angle_closure_glaucoma,,,,benign_prostatic_hyperplasia
t4,toy_eye,"Toy case: sudden painless total loss of vision in one eye an hour ago.",central_retinal_artery_occlusion,,,,plantar_fasciitis
t5,toy_eye,"Toy case: red eye with thick yellow discharge, lids stuck together this morning.",bacterial_conjunctivitis,,,,ingrown_toenail
t6,toy_eye,"Toy case: scratched my eye on a branch, now it waters and hurts to open.",corneal_abrasion,,,,ingrown_toenail


Writing example_cases.csv


In [6]:
%%writefile example_vague.csv
input_id,text
v1,"Toy input: my eye feels kind of off."
v2,"Toy input: something is not right, hard to explain."
v3,"Toy input: I just don't feel good today."


Writing example_vague.csv


In [7]:
!rm -f mock.jsonl
!python run.py --model mock --cases example_cases.csv --options example_options.json --vague example_vague.csv --out mock.jsonl
!python analyze.py mock.jsonl

91 jobs to run for mock (0 already done)
  50/91
# Results summary

Harness commit(s): 73c2c48
Resolved model versions: mock=mock-0

## Output resolution (Amendment 1, descriptive)

- mock: exact-zero rate 0.0%; values on a 0.01 grid 0.0%; smallest nonzero 0.0011 (n=739)

## mock / toy_eye
- H0: 60/60 repeat pairs byte-identical; TVD noise floor (p95) = 0.0000; log-ratio floor = 0.0000
- H1: violation rate 100.0% [61.0%, 100.0%] (n=6); top-1 flip rate 80.0%; mean max-TVD 0.626; Wilcoxon p=0.0156
- H2: violation rate 100.0% [20.7%, 100.0%] (n=1); top-1 flip rate 0.0%; mean max-TVD 0.365
- D1 matched H1 (deviation): mean excess TVD perm minus repeat 0.4940 [0.4036, 0.5937]; excess top-1 flip rate +80.0%; two-sided Wilcoxon p=0.0312 (n=6)
- H3: violation rate 100.0% [61.0%, 100.0%] (n=6); mean distractor mass 0.0990; mean log-ratio shift 2.375
- H4: normalized entropy vague 0.680 vs informative 0.771; P(vague lower) test p=0.131; mean max-prob on vague 0.545
- H4N (exploratory): mean mass

## 5. Live checks (one Jev call, one Opus call; a few cents total)

In [8]:
import json
from collections import OrderedDict
from clients import JevClient, OpusClient
opts = OrderedDict(sorted(json.load(open('example_options.json'))['toy_eye']['options'].items()))
state = 'New flashes and a dark curtain coming over the vision in my right eye since yesterday.'
for name, client in (('Jev', JevClient()), ('Opus', OpusClient('verbalized'))):
    p, raw = client.classify(state, opts)
    print(f"{name} ({raw.get('resolved_model')}):")
    for k, v in sorted(p.items(), key=lambda x: -x[1])[:4]:
        print(f'  {v:.3f}  {k}')

Jev (typesafe/jev-1.13-20260917):
  0.910  retinal_detachment
  0.090  posterior_vitreous_detachment
  0.000  acute_angle_closure_glaucoma
  0.000  anterior_uveitis
Opus (claude-opus-5-5):
  0.904  retinal_detachment
  0.060  posterior_vitreous_detachment
  0.020  central_retinal_artery_occlusion
  0.005  acute_angle_closure_glaucoma


## 6. Study data

Files go in `MyDrive/jev_bench/data/`: `cases.csv`, `options.json`, `vague.csv` for the vignette sets, and `release_test_patients.csv`, `release_evidences.json`, `release_conditions.json` for DDXPlus.

In [9]:
def have(*paths):
    missing = [p for p in paths if not os.path.exists(p)]
    if missing:
        print('Skipping: not uploaded yet ->', ', '.join(os.path.basename(m) for m in missing))
    return not missing

sh = get_ipython().system

if have(f'{D}/release_test_patients.csv', f'{D}/release_evidences.json', f'{D}/release_conditions.json'):
    sh(f'python ddxplus_to_cases.py --patients {D}/release_test_patients.csv --evidences {D}/release_evidences.json '
       f'--conditions {D}/release_conditions.json --n 1000 --out {D}/ddxplus_cases.csv --options-out {D}/ddxplus_options.json')
    snapshot('ddxplus-converted')

Skipping: not uploaded yet -> release_test_patients.csv, release_evidences.json, release_conditions.json


## 7. Runs
Only jobs not already in `results.jsonl` run. In v4 that means: 30 new H4N calls each for Jev and Opus verbalized (a couple of minutes), then the fresh `opus_sampled_v2` arm (45 cases x 20 calls, about 40 minutes, and silent until done). A snapshot is committed before each run.

In [10]:
VIG = (f'{D}/cases.csv', f'{D}/options.json', f'{D}/vague.csv')
if have(*VIG):
    snapshot('pre-run-jev')
    sh(f'python run.py --model jev --cases {D}/cases.csv --options {D}/options.json --vague {D}/vague.csv --out {OUT} --max-calls 5000')

snapshot "pre-run-jev": commit 8965c7b
30 jobs to run for jev (570 already done)


In [11]:
if have(*VIG):
    snapshot('pre-run-opus-verbalized')
    sh(f'python run.py --model opus_verbalized --cases {D}/cases.csv --options {D}/options.json --vague {D}/vague.csv --out {OUT} --max-calls 5000')

snapshot "pre-run-opus-verbalized": commit 54988ba
30 jobs to run for opus_verbalized (570 already done)


In [12]:
# v4: parser-fixed sampled arm, recorded as opus_sampled_v2 (the v3 arm stays in the file, unused)
if have(f'{D}/cases.csv', f'{D}/options.json'):
    snapshot('pre-run-opus-sampled-v2')
    sh(f'python run.py --model opus_sampled --cases {D}/cases.csv --options {D}/options.json --out {OUT} --max-calls 5000')

snapshot "pre-run-opus-sampled-v2": commit 478ba55
45 jobs to run for opus_sampled_v2 (0 already done)


In [13]:
if have(f'{D}/ddxplus_cases.csv', f'{D}/ddxplus_options.json'):
    snapshot('pre-run-ddxplus')
    for m in ('jev', 'opus_verbalized'):
        sh(f'python run.py --model {m} --cases {D}/ddxplus_cases.csv --options {D}/ddxplus_options.json --out {OUT} --conditions H0 --max-calls 1100')

Skipping: not uploaded yet -> ddxplus_cases.csv, ddxplus_options.json


## 8. Analysis and final commit

In [14]:
if have(OUT):
    extra = f'--cases {D}/ddxplus_cases.csv' if os.path.exists(f'{D}/ddxplus_cases.csv') else ''
    sh(f'python analyze.py {OUT} {extra} > {WORK}/runs/summary.md')
    print(open(f'{WORK}/runs/summary.md').read())
    snapshot('analysis')
    print(_git('log', '--oneline', '-10').stdout)

# Results summary

Harness commit(s): 478ba55, 54988ba, 6cb2181, 8965c7b, 9ab7e87, ef4e15e
Resolved model versions: jev=typesafe/jev-1.13-20260917, opus_sampled=claude-opus-5-5, opus_sampled_v2=claude-opus-5-5, opus_verbalized=claude-opus-5-5

## Output resolution (Amendment 1, descriptive)

- jev: exact-zero rate 93.8%; values on a 0.01 grid 99.6%; smallest nonzero 0.01 (n=22785)
- opus_sampled: exact-zero rate 97.4%; values on a 0.01 grid 100.0%; smallest nonzero 1 (n=1216)
- opus_sampled_v2: exact-zero rate 97.4%; values on a 0.01 grid 100.0%; smallest nonzero 1 (n=1178)
- opus_verbalized: exact-zero rate 2.4%; values on a 0.01 grid 4.2%; smallest nonzero 0.0003 (n=22785)

## jev / semigran45
- H0: 295/450 repeat pairs byte-identical; TVD noise floor (p95) = 0.0600; log-ratio floor = 0.2895
- H1: violation rate 31.1% [19.5%, 45.7%] (n=45); top-1 flip rate 4.0%; mean max-TVD 0.061; Wilcoxon p=0.926
- D1 matched H1 (deviation): mean excess TVD perm minus repeat 0.0252 [0.0139, 0.0383]

In [15]:
import json
for l in open(OUT):
    r = json.loads(l)
    if r['model'] == 'opus_sampled_v2' and r.get('error'):
        print(r['case_id'], r['error'][:300], '\n')

sg_em02 ValueError("all-zero distribution; example answers: ['', '', '']") 

sg_em03 ValueError("all-zero distribution; example answers: ['', '', '']") 

sg_em07 ValueError("all-zero distribution; example answers: ['', '', '']") 

sg_ne01 ValueError("all-zero distribution; example answers: ['', '', '']") 

sg_ne05 ValueError("all-zero distribution; example answers: ['', '', '']") 

sg_ne07 ValueError("all-zero distribution; example answers: ['', '', '']") 

sg_ne11 ValueError("all-zero distribution; example answers: ['', '', '']") 

sg_ne15 ValueError("all-zero distribution; example answers: ['', '', '']") 

sg_sc02 ValueError("all-zero distribution; example answers: ['', '', '']") 

sg_sc04 ValueError("all-zero distribution; example answers: ['', '', '']") 

sg_sc12 ValueError("all-zero distribution; example answers: ['', '', '']") 

sg_sc13 ValueError("all-zero distribution; example answers: ['', '', '']") 

sg_sc14 ValueError("all-zero distribution; example answers: ['', '', '']") 


In [16]:
import json, anthropic
from clients import _claude_prompt, OPUS_MODEL
cli = anthropic.Anthropic()
opts = json.load(open(f'{D}/options.json'))['semigran45']['options']
case = next(r for r in __import__('csv').DictReader(open(f'{D}/cases.csv')) if r['case_id'] == 'sg_em02')
for mode, mt in (('sampled', 50), ('verbalized', 3000)):
    m = cli.messages.create(model=OPUS_MODEL, max_tokens=mt,
        system="You are a diagnostic classifier. Follow the output format exactly.",
        messages=[{"role": "user", "content": _claude_prompt(case['text'], dict(sorted(opts.items())), mode)}])
    print(mode, '| stop:', m.stop_reason, '| blocks:', [b.type for b in m.content], '| usage:', m.usage)

sampled | stop: max_tokens | blocks: ['thinking'] | usage: Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=716, output_tokens=50, output_tokens_details=OutputTokensDetails(thinking_tokens=50), server_tool_use=None, service_tier='standard')
verbalized | stop: end_turn | blocks: ['thinking', 'text'] | usage: Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=750, output_tokens=582, output_tokens_details=OutputTokensDetails(thinking_tokens=96), server_tool_use=None, service_tier='standard')
